In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

# find project root whether the notebook runs from root or from notebooks/
here = Path.cwd()
ROOT = here if (here / "data" / "raw").exists() else here.parent
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
assert RAW.exists(), f"data/raw not found from {here}"

YEAR = 2025
gdf = gpd.read_file(RAW / f"Latur_GT_ND_{YEAR}.shp")
print("fields:", len(gdf), "| CRS:", gdf.crs, "| columns:", len(gdf.columns))

fields: 769 | CRS: EPSG:4326 | columns: 28


In [2]:
pd.set_option("display.max_rows", None)

summary = []
for c in gdf.columns:
    if c == gdf.geometry.name:
        continue
    s = gdf[c]
    summary.append({
        "column": c,
        "dtype": str(s.dtype),
        "nulls": int(s.isna().sum()),
        "nunique": int(s.nunique(dropna=True)),
        "sample": str(list(s.dropna().unique()[:3]))[:60],
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

In [3]:
# 2025 has some geometry problems (1 empty, 2 broken). Fix what we can, report the rest.

# count problems before
empty_before = gdf.geometry.is_empty.sum() + gdf.geometry.isna().sum()
invalid_before = (~gdf.geometry.is_valid).sum()
print("before -> empty/missing:", int(empty_before), "| invalid:", int(invalid_before))

# repair broken shapes with buffer(0), a standard fix for self-crossing polygons
gdf["geometry"] = gdf.geometry.buffer(0)

# count problems after
empty_after = gdf.geometry.is_empty.sum() + gdf.geometry.isna().sum()
invalid_after = (~gdf.geometry.is_valid).sum()
print("after  -> empty/missing:", int(empty_after), "| invalid:", int(invalid_after))

before -> empty/missing: 1 | invalid: 3
after  -> empty/missing: 1 | invalid: 1


In [4]:
# Build a clean table. Keep everything with real signal, drop only junk.
# 2025 is the clean file: no doubled columns, so no merging needed.

# safety net: turn any "-" or "N/A" in soil description into real blanks (should be none here)
soil_desc = gdf[["SColor", "SStructure", "STexture", "SDepth"]].replace(["-", "N/A"], np.nan)

clean = gpd.GeoDataFrame({
    "uid": gdf["UID"],                 # field ID
    "year": YEAR,                      # which year (2025)

    # location
    "taluka": gdf["Taluka"],
    "village": gdf["Village"],

    # crop info
    "crop": gdf["Crop_Name"],          # already a single clean column
    "variety": gdf["A_Variety"],
    "irrigation": gdf["Irrigation"],

    # target
    "crop_yield": pd.to_numeric(gdf["C_Yeild"], errors="coerce"),   # yield as number

    # dates (text -> real dates)
    "sowing_date": pd.to_datetime(gdf["ASowing"], dayfirst=True, errors="coerce"),
    "harvest_date": pd.to_datetime(gdf["HA_Date"], errors="coerce"),

    # size
    "area_acres": gdf["Area_acres"],   # already a single clean column

    # soil chemistry
    "carbon": gdf["Carbon"],
    "ph": gdf["pH"],
    "ec": gdf["EC"],
    "nitrogen": gdf["Nitrogen"],
    "phosphorus": gdf["Phosphorus"],
    "potassium": gdf["Potassium"],

    # soil physical description
    "soil_color": soil_desc["SColor"],
    "soil_structure": soil_desc["SStructure"],
    "soil_texture": soil_desc["STexture"],
    "soil_depth": soil_desc["SDepth"],

    # map shape (already repaired in the previous cell)
    "geometry": gdf.geometry,
}, crs=gdf.crs)

print("new table size (rows, columns):", clean.shape)
print()
print("blanks in each column:")
print(clean.drop(columns="geometry").isna().sum())

new table size (rows, columns): (769, 22)

blanks in each column:
uid               0
year              0
taluka            0
village           0
crop              0
variety           0
irrigation        0
crop_yield        0
sowing_date       0
harvest_date      0
area_acres        0
carbon            0
ph                0
ec                0
nitrogen          0
phosphorus        0
potassium         0
soil_color        0
soil_structure    0
soil_texture      0
soil_depth        0
dtype: int64


C:\Users\Predator\AppData\Local\Temp\ipykernel_51168\1866808030.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  "harvest_date": pd.to_datetime(gdf["HA_Date"], errors="coerce"),


In [5]:
# Flag WRONG values only (not missing ones). We mark, never delete.
# For 2025 we also flag the geometry problems that could not be repaired.

flags = []
for idx, r in clean.iterrows():
    f = []

    # yield impossible (<=0) or too high for soybean (>20 quintal/acre)
    y = r["crop_yield"]
    if pd.notna(y) and (y <= 0 or y > 20):
        f.append("yield_suspect")

    # sowing date whose year is not this file's year
    sd = r["sowing_date"]
    if pd.notna(sd) and sd.year != YEAR:
        f.append("bad_sowing_year")

    # area zero or negative (impossible)
    a = r["area_acres"]
    if pd.notna(a) and a <= 0:
        f.append("area_suspect")

    # geometry empty or broken (cannot sample satellite data from it later)
    g = r["geometry"]
    if g is None or g.is_empty or (not g.is_valid):
        f.append("bad_geometry")

    flags.append(";".join(f))

clean["qc_flag"] = flags

# summary
print("flagged rows:", (clean["qc_flag"] != "").sum(), "of", len(clean))
print(clean["qc_flag"].value_counts())
print()

flagged = clean[clean["qc_flag"] != ""]
if len(flagged):
    print(flagged[["uid", "village", "crop_yield", "sowing_date", "area_acres", "qc_flag"]].to_string(index=False))
else:
    print("No wrong values found in 2025.")

flagged rows: 25 of 769
qc_flag
                   744
bad_sowing_year     14
area_suspect         7
yield_suspect        3
bad_geometry         1
Name: count, dtype: int64

           uid              village  crop_yield sowing_date  area_acres         qc_flag
7484-8598-6981              Warwada         6.0  2025-07-18   -0.326011    area_suspect
5509-6041-6863                Bheta        20.3  2025-06-19    2.055013   yield_suspect
5516-6049-6785                Bheta        24.0  2025-06-15    1.622924   yield_suspect
7494-8605-6990              Warwada         6.0  2025-06-20   -0.022146    area_suspect
5639-6187-5888                Ashta        10.0  2024-06-12    1.658528 bad_sowing_year
7478-8585-6955                Ashta         6.0  1964-08-23    3.242461 bad_sowing_year
7435-8541-6852              Dapkyal         5.2  2025-06-18   -0.637436    area_suspect
7398-8500-6801             Zari Kh.         9.0  2025-06-18   -0.342377    area_suspect
5954-6538-4758 Chincholi Ballalnat

In [6]:
# Save the cleaned 2025 table to data/interim. Raw stays untouched.
INTERIM.mkdir(parents=True, exist_ok=True)
out = INTERIM / f"clean_{YEAR}.gpkg"
clean.to_file(out, driver="GPKG")

print("saved:", out)
print("rows:", len(clean), "| columns:", clean.shape[1])
print("flagged (wrong values):", (clean["qc_flag"] != "").sum())
print("PII columns present:", [c for c in clean.columns if c in ("farmer", "phone", "agent")])  # should be []

saved: f:\THESIS\mtech_thesis_crop_yield_estimation\data\interim\clean_2025.gpkg
rows: 769 | columns: 23
flagged (wrong values): 25
PII columns present: []


# Data Cleaning Summary: Latur Soybean Ground Truth

## What this covers
Cleaning of the field-level soybean ground truth data for the crop yield thesis.
Four kharif years from Latur district, Maharashtra: 2022, 2023, 2024, 2025.
Source: ground truth shapefiles.

## Result

| Year | Fields | Flagged | Saved file |
|------|--------|---------|------------|
| 2022 | 55 | 0 | clean_2022.gpkg |
| 2023 | 60 | 2 | clean_2023.gpkg |
| 2024 | 323 | 9 | clean_2024.gpkg |
| 2025 | 769 | 25 | clean_2025.gpkg |
| **Total** | **1,207** | **36** | in data/interim |

All four cleaned files share the same 23 columns and are saved as GeoPackage
(.gpkg) in data/interim. Raw files in data/raw are untouched.

## What was done to each year
1. Loaded the shapefile and inspected every column.
2. For 2022, 2023, 2024 (which were two batches merged), merged the doubled
   crop and area columns back into one each. Area in square meters was
   converted to acres (1 acre = 4046.86 m2).
3. Built a clean table keeping only useful columns, with simple lowercase
   names and a year column. Text yield and dates converted to real
   numbers and dates.
4. Dropped junk: private info (farmer, phone, agent), empty columns
   (Copper, Iron, Water pH, etc, all N/A), duplicate columns, and merge
   leftovers (layer, path, fid).
5. For 2025, repaired broken polygons with buffer(0).
6. Flagged wrong values in a qc_flag column. Marked, never deleted.

## Columns kept (23)
uid, year, taluka, village, crop, variety, irrigation, crop_yield,
sowing_date, harvest_date, area_acres, carbon, ph, ec, nitrogen,
phosphorus, potassium, soil_color, soil_structure, soil_texture,
soil_depth, geometry, qc_flag

## Key findings
- All 1,207 fields have a yield value. No missing targets.
- All four years are EPSG:4326, all soybean, all Latur district.
- The 1,207 fields are all distinct. No field repeats across years.
- Missing values are tiny once pooled: variety and sowing date about 2.8%,
  soil texture about 1.8%. All left as blanks for now.
- Soil description (color, texture, depth) is weak in 2022 (40% missing)
  and 2024 (63% missing), but full in 2023 and 2025. Keep or drop decision
  deferred to EDA.
- Village Harwadi (2025) has several bad polygons (negative area, broken
  geometry).

## Flagged values (36 total, kept in files)
- yield_suspect: impossible or very high yields. Includes 720 and 260 in
  2023 (impossible), a 0 in 2025, and several high values 20 to 47 in 2024.
- bad_sowing_year: sowing date year does not match the file year (typos like
  1960, 1964, or wrong crop year). Yield is fine, only the date is wrong.
- area_suspect: zero or negative area (broken polygons).
- bad_geometry: empty or broken shapes that could not be repaired (2025).
